# Implementazione EME — prime verifiche

Parto dal codice della repository ufficiale e provo a ricostruire gradualmente l'architettura su YOLOv5-Small. In questa fase lavoro con input sintetici: l'obiettivo è controllare che i moduli siano collegati correttamente, non valutare ancora le predizioni.

I file `yolov5s_eme.yaml`, `eme_weak_sup.py` e `common.py` si trovano in `yolov5/models` e sono gestiti tramite Git.

### Impostazione della directory di lavoro

Individuo la cartella principale della repository e imposto `yolov5` come directory di lavoro, così da usare gli stessi percorsi relativi sul PC e sulla workstation.

In [ ]:
from pathlib import Path
import os
import sys

# Ricerca della repository a partire dalla directory corrente
root = Path.cwd().resolve()
while not (root / "yolov5" / "models" / "yolo.py").is_file():
    if root == root.parent:
        raise FileNotFoundError("Aprire il notebook dalla repository EME.")
    root = root.parent

yolo_dir = root / "yolov5"
os.chdir(yolo_dir)
if str(yolo_dir) not in sys.path:
    sys.path.insert(0, str(yolo_dir))

print("Directory di lavoro:", Path.cwd())
print("Interprete Python:", sys.executable)
print("Versione Python:", sys.version.split()[0])

files = [
    "train_rgbt_earlyFusion.py",
    "train_rgbt_distillation_earlyFusion.py",
    "models/yolov5s_eme.yaml",
    "models/eme_weak_sup.py",
    "models/common.py",
    "requirements.txt"
]

for file in files:
    print(file, "OK" if Path(file).is_file() else "MANCANTE")

### Dipendenze e caricamento del modello

Importo la classe `Model` di YOLOv5. Le dipendenze, tra cui `kornia` per il modulo di early fusion, devono essere già installate nell’ambiente Python selezionato come kernel in VS Code.

In [ ]:
from models.yolo import Model

### Configurazione dello student

Istanzio YOLOv5-Small dal file YAML che ho adattato per EME: 6 canali di ingresso (RGB + termico), 6 classi e `earlyFusionConv` come primo modulo. Il file YAML è esterno al notebook.

In [ ]:
model = Model(cfg="models/yolov5s_eme.yaml")

### Verifica del primo modulo

Controllo la struttura del layer iniziale e i parametri letti dal file YAML, per essere sicuro che il modello stia usando davvero ShaPE e non il normale primo layer di YOLOv5.

In [ ]:
print(model.model[0]) #shape

In [ ]:
print("Canali di ingresso:", model.yaml["ch"])
print("Numero di classi:", model.yaml["nc"])
print("Primo modulo:", type(model.model[0]).__name__)

### Layer del backbone da utilizzare

Individuo i livelli del backbone che mi interessano per WeakSup. Scelgo i layer 4, 6 e 9 perché forniscono feature a tre scale diverse (1/8, 1/16 e 1/32); controllo anche quanti canali hanno. È un adattamento a YOLOv5 dei livelli usati nell'implementazione originale su ResNet.

In [ ]:
for i in [4, 6, 9, 17, 20, 23, 24]:
    print(i, model.model[i].type)

In [ ]:
print("Layer 4:", model.model[4].cv3.conv.out_channels)
print("Layer 6:", model.model[6].cv3.conv.out_channels)
print("Layer 9:", model.model[9].cv2.conv.out_channels)

### Prima prova della Classification Head

La testa di classificazione applica un global average pooling e un layer lineare. Qui uso una feature casuale con 128 canali per verificare soltanto che l'output abbia un valore per ciascuna delle 6 classi.

In [ ]:
from models.eme_weak_sup import EMEClassificationHead
classification_head = EMEClassificationHead(
    in_channels=128,
    num_classes=6
)

import torch

x = torch.randn(2, 128, 80, 80)

output = classification_head(x)

print("Input:", x.shape)
print("Output:", output.shape)

### Prima prova della Segmentation Head

La testa di segmentazione deve invece mantenere l'informazione spaziale: da una feature `[2, 128, 80, 80]` mi aspetto 6 mappe `[80, 80]` per immagine. Per ora sono solo output non addestrati.

In [ ]:
from models.eme_weak_sup import EMESegmentationHead

segmentation_head = EMESegmentationHead(
    in_channels=128,
    num_classes=6
)

x = torch.randn(2, 128, 80, 80)
output = segmentation_head(x)

print(output.shape)

### Recuperare una feature intermedia con un hook

Registro un `forward hook` sul layer 4. Ogni volta che quel layer viene eseguito, la funzione salva il suo output nel dizionario `features`, senza dover modificare il `forward` originale di YOLOv5.

In [ ]:
features = {}

def salva_feature(modulo, input, output):
    features["layer4"] = output

hook = model.model[4].register_forward_hook(salva_feature)

### Test dell'hook

Eseguo un passaggio con un input RGB-T sintetico di `256 × 256` pixel. Il layer 4 produce una feature a risoluzione 1/8, quindi `32 × 32`. Dopo il test rimuovo l'hook: la feature già salvata resta disponibile, ma le esecuzioni successive non la aggiorneranno.

In [ ]:
model.eval()

device = next(model.parameters()).device
x = torch.rand(1, 6, 256, 256, device=device)

with torch.no_grad():
    _ = model(x)

hook.remove()

print("Feature layer 4:", features["layer4"].shape)

### Collegamento delle teste alla feature reale

Ora passo alle teste la feature estratta dal layer 4, invece di crearla casualmente. È un primo controllo del collegamento con il backbone: qui uso `eval()` e `no_grad()`, quindi non sto effettuando alcun addestramento.

In [ ]:
feat = features["layer4"]
classification_head = classification_head.to(device).eval()
segmentation_head = segmentation_head.to(device).eval()

with torch.no_grad():
    cls_output = classification_head(feat)
    seg_output = segmentation_head(feat)

print("Classification:", cls_output.shape)
print("Segmentation:", seg_output.shape)

### WeakSup su tre livelli del backbone

Passo dalla singola feature alle tre scale scelte. Gli hook raccolgono gli output dei layer 4, 6 e 9; il modulo `EMEWeakSupHeads` usa tre convoluzioni 1×1 per adattare i canali e applicare le due teste condivise.

Da qui uso input di `512 × 640` pixel, cioè la risoluzione indicata negli esperimenti di EME. Anche in questo caso non sto ancora caricando il dataset né calcolando le loss.

In [ ]:
import importlib
import models.eme_weak_sup as weak_sup

importlib.reload(weak_sup)

EMEWeakSupHeads = weak_sup.EMEWeakSupHeads

features = {}
handles = []

def make_hook(layer_id):
    def hook(module, inputs, output):
        features[layer_id] = output
    return hook

for layer_id in [4, 6, 9]:
    handle = model.model[layer_id].register_forward_hook(
        make_hook(layer_id)
    )
    handles.append(handle)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device).eval()

weak_heads = EMEWeakSupHeads(
    num_classes=6
).to(device).eval()

x = torch.rand(1, 6, 512, 640, device=device)

with torch.no_grad():
    _ = model(x)

    cls_outputs, seg_outputs = weak_heads([
        features[4],
        features[6],
        features[9]
    ])



### Controllo delle dimensioni in uscita

Verifico che, per ciascun livello, la classificazione restituisca 6 valori e la segmentazione una mappa per classe. Le risoluzioni attese sono `64 × 80`, `32 × 40` e `16 × 20`.

In [ ]:
for i in range(3):
    print(f"Livello {i}:")
    print(" Classification:", cls_outputs[i].shape)
    print(" Segmentation:  ", seg_outputs[i].shape)

### Rimozione degli hook

Finita la verifica, scollego gli hook registrati. Se dovessi ripetere il test in una nuova esecuzione, dovrò registrarli di nuovo.

In [ ]:
for handle in handles:
    handle.remove()